In [ ]:


# FINAL MULTI-DOMAIN OIL SPILL TRAINING (STABLE + RECALL BOOST

!pip install -q segmentation_models_pytorch albumentations tqdm

import os
import cv2
import torch
import numpy as np
import albumentations as A
import segmentation_models_pytorch as smp
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2
import torch.optim as optim

# ============================================================
# CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 384
BATCH = 3
EPOCHS = 25
LR = 6e-5

TRAIN_DIR = "/kaggle/working/combined_images"
TRAIN_MASK_DIR = "/kaggle/working/combined_masks"

VALID_DIR = "/kaggle/input/datasets/mukeshvarmaa/newseg/newseg/valid"
VALID_MASK_DIR = "/kaggle/working/masks_valid_multi"

MODEL_PATH = "/kaggle/working/final_multi_domain_model.pth"

print("Device:", DEVICE)
print("Train images:", len(os.listdir(TRAIN_DIR)))
print("Train masks :", len(os.listdir(TRAIN_MASK_DIR)))

# ============================================================
# DATASET
# ============================================================

class OilDataset(Dataset):
    def __init__(self, img_dir, mask_dir, train=True):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = os.listdir(mask_dir)

        if train:
            self.transform = A.Compose([
                A.Resize(IMG_SIZE, IMG_SIZE),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ColorJitter(p=0.4),
                A.GaussianBlur(p=0.2),
                A.RandomGamma(p=0.3),
                A.Normalize(),
                ToTensorV2()
            ])
        else:
            self.transform = A.Compose([
                A.Resize(IMG_SIZE, IMG_SIZE),
                A.Normalize(),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        name = self.files[i]
        img_name = name.replace(".png", ".jpg")

        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.mask_dir, name), 0)

        aug = self.transform(image=img, mask=mask)
        return aug["image"], aug["mask"].long()



model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    decoder_attention_type="scse",
    in_channels=3,
    classes=3
).to(DEVICE)


# Heavier oil emphasis
class_weights = torch.tensor([0.2, 1.2, 0.3]).to(DEVICE)

ce_loss = torch.nn.CrossEntropyLoss(weight=class_weights)

tversky = smp.losses.TverskyLoss(
    mode="multiclass",
    alpha=0.4,   # lower FP penalty
    beta=0.6     # higher FN penalty (boost recall)
)

def loss_fn(pred, mask):
    return 0.5 * ce_loss(pred, mask) + 0.5 * tversky(pred, mask)

optimizer = optim.AdamW(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)



train_ds = OilDataset(TRAIN_DIR, TRAIN_MASK_DIR, True)
valid_ds = OilDataset(VALID_DIR, VALID_MASK_DIR, False)

train_ld = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2)
valid_ld = DataLoader(valid_ds, BATCH, num_workers=2)

torch.cuda.empty_cache()
best_val = 999
early_stop_counter = 0

for ep in range(EPOCHS):

    model.train()
    train_loss = 0

    for x, y in tqdm(train_ld):
        x, y = x.to(DEVICE), y.to(DEVICE)

        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in valid_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            val_loss += loss_fn(pred, y).item()

    val_loss /= len(valid_ld)
    scheduler.step()

    print(f"\nEpoch {ep+1}")
    print("Train Loss:", train_loss/len(train_ld))
    print("Val Loss  :", val_loss)
    print("LR:", scheduler.get_last_lr()[0])

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("✅ Saved Best Model")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if early_stop_counter >= 5:
        print("⛔ Early stopping triggered")
        break

print("\nTraining Complete.")

In [ ]:
# ============================================================
# LOAD MODEL + TEST
# ============================================================

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 384


MODEL_PATH = "/kaggle/working/final_multi_domain_model.pth"

TEST_DIR = "/kaggle/input/datasets/mukeshvarmaa/newseg/newseg/test"



model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights=None,
    decoder_attention_type="scse",
    in_channels=3,
    classes=3
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("Model loaded successfully")

#============================================================
# TRANSFORM
# ============================================================

transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2()
])

# ============================================================
# TEST FUNCTION
# ============================================================

def test_model(image_path, min_area=80):

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    H, W, _ = img.shape

    aug = transform(image=img)
    x = aug["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(x)
        pred = torch.argmax(pred, dim=1)[0].cpu().numpy()

    mask = cv2.resize(pred.astype(np.uint8), (W,H), interpolation=cv2.INTER_NEAREST)

    oil_mask = (mask == 1).astype(np.uint8)

    # remove small noise
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(oil_mask, 8)
    clean = np.zeros_like(oil_mask)

    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > min_area:
            clean[labels==i] = 1

    overlay = img.copy()
    overlay[clean==1] = [255,0,0]

    result = cv2.addWeighted(img,0.7,overlay,0.3,0)

    plt.figure(figsize=(14,4))

    plt.subplot(1,3,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(clean,cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(result)
    plt.title("Oil Detection")
    plt.axis("off")

    plt.show()

# ============================================================
# TEST MULTIPLE IMAGES
# ============================================================

for img_name in os.listdir(TEST_DIR)[:6]:
    print("Testing:", img_name)
    test_model(os.path.join(TEST_DIR, img_name))